In [ ]:
# File: scrape_final_robust_fixed.py

import pandas as pd
import time
import sys
import logging

# Attempt to import necessary libraries
try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.keys import Keys
    from selenium.webdriver.chrome.service import Service as ChromeService
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
    from webdriver_manager.chrome import ChromeDriverManager
except ImportError:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    logging.error("Critical libraries not found. Please run 'pip install selenium pandas webdriver-manager' in your terminal.")
    sys.exit(1)


def handle_popups(driver, wait):
    """
    Checks for and closes the login/subscription pop-up dialog.
    This function is non-blocking and will not fail if the pop-up is not present.
    """
    try:
        # This is the specific locator for the pop-up's close button
        dialog_close_locator = (By.CSS_SELECTOR, 'div.el-dialog__wrapper button.el-dialog__headerbtn')
        # Use a short wait time to quickly check for the pop-up
        short_wait = WebDriverWait(driver, 2)
        close_button = short_wait.until(EC.element_to_be_clickable(dialog_close_locator))
        logging.info("Login/subscription pop-up found. Attempting to close it.")
        close_button.click()
        time.sleep(1)  # Give a moment for the dialog to disappear
    except TimeoutException:
        # This is expected if the pop-up is not present. Do nothing.
        logging.info("No pop-up dialog found. Continuing...")
    except Exception as e:
        logging.warning(f"An error occurred while trying to close the pop-up: {e}")


def scrape_paginated_table_robust_final():
    """
    Scrapes a paginated table with robust pop-up handling and click strategies.
    """
    # --- FIX 1: CORRECTED THE TARGET URL ---
    #target_url = "https://www.gurufocus.com/economic_indicators/57/sp-500-pe-ratio"
    target_url = "https://www.gurufocus.com/economic_indicators/6778/nasdaq-100-pe-ratio"

    table_locator = (By.ID, "non-sticky-table")
    next_button_locator = (By.CSS_SELECTOR, "button.btn-next")
    # This locator will be used to detect when a page is loading new data
    loading_mask_locator = (By.CSS_SELECTOR, "div.el-loading-mask")

    options = webdriver.ChromeOptions()
    # Running in headed mode is often more stable for complex sites like this
    # options.add_argument("--headless") 
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_experimental_option('excludeSwitches', ['enable-logging'])

    driver = None
    try:
        logging.info("Initializing WebDriver...")
        service = ChromeService(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)
        logging.info("WebDriver initialized successfully.")
        
        wait = WebDriverWait(driver, 20) # A reasonable default wait time

        logging.info("Navigating to URL: %s", target_url)
        driver.get(target_url)

        # Initial check for any pop-ups on page load
        handle_popups(driver, wait)

        logging.info("Waiting for the data table to become visible...")
        table_element = wait.until(EC.visibility_of_element_located(table_locator))
        
        header_elements = table_element.find_elements(By.XPATH, ".//thead//th")
        header = [h.text.strip() for h in header_elements if h.text.strip()]
        logging.info("Scraped table header: %s", header)
        
        all_rows_data = []
        page_number = 1

        while True:
            logging.info("Starting scrape for page %d.", page_number)
            
            # --- FIX 3 (Part 1): Wait for loading mask to disappear before scraping ---
            # This ensures the data on the page is stable and not from the previous page.
            wait.until(EC.invisibility_of_element_located(loading_mask_locator))
            
            # Wait for at least one row to be present in the table body
            wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="non-sticky-table"]//tbody/tr')))
            table_rows = driver.find_elements(By.XPATH, '//*[@id="non-sticky-table"]//tbody/tr')
            logging.info("Found %d rows on page %d.", len(table_rows), page_number)
            
            for row in table_rows:
                row_data = [cell.text for cell in row.find_elements(By.TAG_NAME, "td")]
                if len(row_data) == len(header):
                    all_rows_data.append(row_data)
                else:
                    logging.warning("Row data length (%d) doesn't match header length (%d). Skipping row: %s", len(row_data), len(header), row_data)

            try:
                # --- FIX 2: PROACTIVE POP-UP HANDLING ---
                # Before interacting with the 'Next' button, check for and close any pop-ups.
                handle_popups(driver, wait)

                next_button = driver.find_element(*next_button_locator)
                if "disabled" in next_button.get_attribute("class"):
                    logging.info("Last page reached (Next button is disabled).")
                    break

                logging.info("Scrolling to and clicking 'Next' button...")
                
                # --- FIX 3 (Part 2): USE A JAVASCRIPT CLICK FOR ROBUSTNESS ---
                # This is less likely to be intercepted than a standard Selenium click.
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_button)
                time.sleep(0.5) # A brief pause can help stability
                driver.execute_script("arguments[0].click();", next_button)

                page_number += 1

            except NoSuchElementException:
                logging.info("No more 'Next' button found. Assuming end of pages.")
                break
            except Exception as e:
                logging.error(f"An unexpected error occurred while trying to navigate to the next page: {e}")
                break
        
        df = pd.DataFrame(all_rows_data, columns=header)
        return df

    except Exception as e:
        logging.exception("An unexpected error occurred during the scraping process.")
        return None
    finally:
        if driver:
            logging.info("Closing the browser session.")
            driver.quit()

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    
    logging.info("Starting the web scraping process.")
    scraped_df = scrape_paginated_table_robust_final()

    if scraped_df is not None and not scraped_df.empty:
        logging.info("Scraping process completed successfully.")
        csv_filename = "gurufocus_sp500_pe_ratio.csv"
        scraped_df.to_csv(csv_filename, index=False)
        logging.info("Data successfully saved to '%s'", csv_filename)
        print(f"\nSuccess! Data saved to {csv_filename}")
        print(scraped_df.head())
    else:
        logging.error("Scraping failed. Please review the logs.")

In [ ]:
# File: scrape_final_robust.py

import pandas as pd
import time
import sys
import logging

# Attempt to import necessary libraries
try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.keys import Keys # <-- FIX: Import the Keys class
    from selenium.webdriver.chrome.service import Service as ChromeService
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
    from webdriver_manager.chrome import ChromeDriverManager
except ImportError:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    logging.error("Critical libraries not found. Please run 'pip3 install selenium pandas webdriver-manager' in your terminal.")
    sys.exit(1)


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
"""
Scrapes a paginated table with maximum stability by running in headed mode
and handling potential pop-up interruptions.
"""
target_url = "https://www.gurufocus.com/economic_indicators/6778/nasdaq-100-pe-ratio" 

table_locator = (By.ID, "non-sticky-table")
next_button_locator = (By.CSS_SELECTOR, "button.btn-next")
cookie_button_locator = (By.XPATH, "//button[contains(text(), 'Got it')]") # Locator for the cookie banner
dialog_close_locator = (By.CSS_SELECTOR, 'button.el-dialog__headerbtn[aria-label="Close"]')

options = webdriver.ChromeOptions()

# --- SOLUTION 1: RUN IN HEADED (NON-HEADLESS) MODE ---
# The --headless argument is removed. A browser window will now open.
# options.add_argument("--headless") 

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_experimental_option('excludeSwitches', ['enable-logging'])

driver = None
logging.info("Initializing WebDriver...")
service = ChromeService(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
logging.info("WebDriver initialized successfully.")

# --- SOLUTION 3: INCREASE TIMEOUT DURATION ---
wait = WebDriverWait(driver, 45) # Increased timeout to 45 seconds
short_wait = WebDriverWait(driver, 3) # Increased timeout to 45 seconds

logging.info("Navigating to URL: %s", target_url)
driver.get(target_url)

# --- SOLUTION 2: HANDLE COOKIE CONSENT BANNER ---
'''
try:
    logging.info("Looking for a cookie consent banner...")
    cookie_button = wait.until(EC.element_to_be_clickable(cookie_button_locator))
    logging.info("Found cookie banner. Clicking 'Got it'...")
    cookie_button.click()
except TimeoutException:
    logging.info("No cookie consent banner found, or it was not clickable. Continuing...")
'''

logging.info("Waiting for the data table to become visible...")
table_element = wait.until(EC.visibility_of_element_located(table_locator))

header_elements = table_element.find_elements(By.XPATH, ".//thead//th")
header = [h.text.strip() for h in header_elements if h.text.strip()]
logging.info("Scraped table header: %s", header)


In [ ]:
from detonator import mongo_2_df
import pandas as pd
TICKER = 'TSLA'
dailies = TickerDailyInfo.objects(ticker=TICKER).order_by('-trade_date')
dailies_df = mongo_2_df(dailies)
dailies_df['trade_date'] = pd.to_datetime(dailies_df['trade_date'], format='%Y,%m,%d,%H,%M,%S,%f')
dailies_df.set_index('trade_date', inplace=True)
dailies_df.sort_index(ascending=True, inplace=True)
print(dailies_df.shape)
dailies_df = dailies_df[['ticker','open', 'high', 'low', 'close', 'volume']]
dailies_df.dropna(inplace=True)
dailies_df.to_csv(f'{TICKER}.csv')
df = pd.read_csv(f'{TICKER}.csv', index_col=0, parse_dates=True)
df


In [ ]:

logging.info("Starting scrape for page %d.", page_number)
wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="non-sticky-table"]//tbody/tr')))
tables = driver.find_elements(By.ID, 'non-sticky-table')
#first_row_of_current_page = table_rows[0]
#for row in table_rows:
#    all_rows_data.append([cell.text for cell in row.find_elements(By.TAG_NAME, "td")])


In [ ]:
table = tables[0]
table_rows = table.find_elements(By.XPATH, '//*[@id="non-sticky-table"]//tbody/tr')
for row in table_rows:
    print([cell.text for cell in row.find_elements(By.TAG_NAME, "td")])

In [ ]:
table_rows = driver.find_elements(By.XPATH, '//*[@id="non-sticky-table"]//tbody/tr')
for row in table_rows:
    print([cell.text for cell in row.find_elements(By.TAG_NAME, "td")])

In [ ]:

all_rows_data = []
page_number = 1

# The rest of the scraping loop remains the same
while True:
    # (Loop logic as before)
    logging.info("Starting scrape for page %d.", page_number)
    wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="non-sticky-table"]//tbody/tr')))
    table_rows = driver.find_elements(By.XPATH, '//*[@id="non-sticky-table"]//tbody/tr')
    logging.info("Found %d rows on page %d.", len(table_rows), page_number)
    #first_row_of_current_page = table_rows[0]
    for row in table_rows:
        all_rows_data.append([cell.text for cell in row.find_elements(By.TAG_NAME, "td")])


    # --- NEW: SEND ESCAPE KEY ---
    try:
        logging.info("Sending 'Escape' key to dismiss any other potential pop-ups.")
        driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.ESCAPE)
        time.sleep(0.5) # Short pause to allow UI to settle
    except Exception as e:
        logging.error("Could not send ESC key. Error: %s", e)

    try:
        next_button = driver.find_element(*next_button_locator)
        if next_button.get_attribute("disabled"):
            logging.info("Last page reached.")
            break
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_button)
        time.sleep(0.5)
        logging.info("Clicking 'Next' button...")
        next_button.click()
        logging.info("Go to next page...")
        #logging.info("Waiting for page content to refresh...")
        #wait.until(EC.staleness_of(first_row_of_current_page))
        page_number += 1
    except (StaleElementReferenceException, TimeoutException):
        logging.info("No more 'Next' button found. Assuming end of pages.")
        break


In [ ]:

# df = pd.DataFrame(all_rows_data, columns=header)

if driver:
    logging.info("Closing the browser session.")
    driver.quit()


# df

In [ ]:
import os

from datetime import datetime

from yfinance import Ticker

aapl = Ticker('LI')
his = aapl.history(start=datetime(year=2024, month=1, day=6), end='2024-01-13')

In [ ]:
his

In [ ]:
his = aapl.history(period='5d', interval='1m', raise_errors=True)
his

In [ ]:
his.info()

In [ ]:
from dataminer.models import IndexTickers

queries = {
    'index_name': 'spx',
    'as_of_date__gte': '20240101'
}
it = IndexTickers.objects(__raw__ = queries)

In [ ]:
it.first().tickers

In [ ]:
import pandas as pd

csvs = pd.read_csv('https://www.ishares.com/us/products/239708/ishares-russell-1000-value-etf/1467271812596.ajax?fileType=csv&fileName=IWD_holdings&dataType=fund', skiprows = 9)
csvs = csvs[csvs['Asset Class']=='Equity']

In [ ]:
csvs

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
from urllib.parse import urljoin


import pandas as pd
from pandas import DataFrame


def get_ishares_holdings_link(url):
    """
    Requests a given iShares ETF page and extracts the "Detailed Holdings and Analytics"
    link's label and href.

    Args:
        url (str): The URL of the iShares ETF page.

    Returns:
        tuple: A tuple containing (link_label, href_link) if found,
               otherwise (None, None).
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    except requests.exceptions.RequestException as e:
        print(f"Error requesting the URL: {e}")
        return None, None

    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the <a> tag with the specific class and text content
    # We can use a dictionary to specify attributes and their values
    # The `string` argument can be used to match the text content
    link_tag = soup.find(
        'a',
        # class_='icon-xls-export',
        string='Detailed Holdings and Analytics'
    )

    if link_tag:
        link_label = link_tag.get_text(strip=True)
        href_link = link_tag.get('href')

        # iShares often provides relative URLs for these links.
        # We need to construct a full URL if it's relative.
        if href_link and not href_link.startswith(('http://', 'https://')):
            # Assume it's a relative path to the base domain
            base_url = url.split('/us/products/')[0] + '/'
            href_link = urljoin(base_url, href_link)

        return link_label, href_link
    else:
        print("Link 'Detailed Holdings and Analytics' not found on the page.")
        return None, None

def download_csv_from_link(csv_url:str, filename:str="holdings.csv"):
    """
    Downloads a CSV file from a given URL.

    Args:
        csv_url (str): The URL of the CSV file.
        filename (str): The name to save the downloaded CSV file as.
    """
    if not csv_url:
        print("No CSV URL provided for download.")
        return

    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36'
    }

    print(f"Attempting to download CSV from: {csv_url}")
    try:
        csv_response = requests.get(csv_url, headers=headers, stream=True)
        csv_response.raise_for_status() # Check for HTTP errors

        # Ensure the directory exists if specified in filename
        output_dir = os.path.dirname(filename)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)

        with open(filename, 'wb') as f:
            for chunk in csv_response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"CSV file '{filename}' downloaded successfully!")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the CSV: {e}")
    except IOError as e:
        print(f"Error writing the CSV file to disk: {e}")

# --- Main execution ---
target_url = "https://www.ishares.com/us/products/239706/ishares-russell-1000-growth-etf"
target_url = 'https://www.ishares.com/us/products/239710/ishares-russell-2000-etf'

print(f"Requesting page: {target_url}")
label, href = get_ishares_holdings_link(target_url)

if label and href:
    print(f"Found Link Label: '{label}'")
    print(f"Found Href: '{href}'")

    # You can now proceed to download the CSV if needed
    # For demonstration, I'll save it as 'IWB_holdings.csv'

    csvs = pd.read_csv(href, skiprows = 9)
    csvs = csvs[csvs['Asset Class']=='Equity']
    print(csvs)
else:
    print("Failed to find the detailed holdings link.")